# Melbourne Parking Data — Inspect & Clean

Five source files:

| File | Description |
|------|-------------|
| `on-street-parking-bays.csv` | Physical bay inventory with coordinates (~29 k rows) |
| `on-street-parking-bay-sensors.csv` | Real-time sensor occupancy (~3.4 k rows) |
| `parking-zones-linked-to-street-segments.csv` | Zone → street mapping (~926 rows) |
| `pay-stay-zones-linked-to-street-segments.csv` | Pay-and-stay zone → street mapping (~935 rows) |
| `sign-plates-located-in-each-parking-zone.csv` | Restriction rules per zone (~1 821 rows) |

In [118]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore', category=pd.errors.DtypeWarning)

DATA = '../data/'

---
## 0. View Problem Rows — Missing Values & Duplicates
Load all files then use `show_missing()` and `show_duplicates()` to see exactly which rows have issues.

In [164]:
bayres_raw  = pd.read_csv(DATA + 'on-street-car-park-bay-restrictions.csv')
bays_raw    = pd.read_csv(DATA + 'on-street-parking-bays.csv')
sensors_raw = pd.read_csv(DATA + 'on-street-parking-bay-sensors.csv')
zones_raw   = pd.read_csv(DATA + 'parking-zones-linked-to-street-segments.csv')
paystay_raw = pd.read_csv(DATA + 'pay-stay-zones-linked-to-street-segments.csv')
signs_raw   = pd.read_csv(DATA + 'sign-plates-located-in-each-parking-zone.csv')
print('All files loaded.')

All files loaded.


In [120]:
def show_missing(df, name):
    """Show per-column missing counts then display every row that has at least one null."""
    mask = df.isnull().any(axis=1)
    counts = df.isnull().sum()
    counts = counts[counts > 0]
    print(f"{'='*60}")
    print(f"  {name}  —  {mask.sum()} rows with missing values")
    print(f"{'='*60}")
    if counts.empty:
        print('  No missing values.\n')
        return
    print('  Missing counts per column:')
    for col, n in counts.items():
        print(f'    {col:<45} {n:>6}  ({n/len(df)*100:.1f}%)')
    print()
    display(df[mask])


def show_duplicates(df, name, subset=None):
    """
    Display duplicate rows.  keep=False marks ALL copies of a duplicate so you
    can see every instance side-by-side.
    subset: list of column names to check (default = all columns).
    """
    dup_mask = df.duplicated(subset=subset, keep=False)
    label = f'columns {subset}' if subset else 'all columns'
    print(f"{'='*60}")
    print(f"  {name}  —  {dup_mask.sum()} duplicate rows (by {label})")
    print(f"{'='*60}")
    if dup_mask.sum() == 0:
        print('  No duplicates.\n')
        return
    sort_cols = subset if subset else df.columns.tolist()
    display(df[dup_mask].sort_values(sort_cols))


print('Helper functions ready.')

Helper functions ready.


### Missing values — all files

In [166]:
show_missing(bayres_raw, 'on-street-car-park-bay-restrictions.csv')

  on-street-car-park-bay-restrictions.csv  —  4263 rows with missing values
  Missing counts per column:
    Description2                                    1401  (32.9%)
    Description3                                    3679  (86.3%)
    Description4                                    4046  (94.9%)
    Description5                                    4181  (98.1%)
    Description6                                    4238  (99.4%)
    DisabilityExt2                                  1401  (32.9%)
    DisabilityExt3                                  3679  (86.3%)
    DisabilityExt4                                  4046  (94.9%)
    DisabilityExt5                                  4181  (98.1%)
    DisabilityExt6                                  4236  (99.4%)
    Duration2                                       1401  (32.9%)
    Duration3                                       3679  (86.3%)
    Duration4                                       4046  (94.9%)
    Duration5                        

,BayID,DeviceID,Description1,Description2,Description3,Description4,Description5,Description6,DisabilityExt1,DisabilityExt2,...,ToDay3,ToDay4,ToDay5,ToDay6,TypeDesc1,TypeDesc2,TypeDesc3,TypeDesc4,TypeDesc5,TypeDesc6
0,2313,25054,1P MTR M-F 9:30-18:30,2P MTR M-SAT 6:30PM-8:30PM,1P MTR SAT 7.30-6.30PM,1P SUN 7:30-18:30,NaN,NaN,120,240.0,...,6.0,0.0,NaN,NaN,1P Meter,2P Meter,1P Meter,1P,NaN,NaN
1,5996,28891,2P MTR M-SAT 7:30-20:30,2P SUN 7:30-18:30,NaN,NaN,NaN,NaN,240,240.0,...,NaN,NaN,NaN,NaN,2P Meter,2P,NaN,NaN,NaN,NaN
2,6844,27079,4P TKT A M-F 7:30-18:30,4P TKT A SAT 7:30-12:30,NaN,NaN,NaN,NaN,480,480.0,...,NaN,NaN,NaN,NaN,4P Ticket A,4P Ticket A,NaN,NaN,NaN,NaN
3,6103,27054,2P MTR M-SAT 7:30-18:30,2P SUN 7:30-18:30,NaN,NaN,NaN,NaN,240,240.0,...,NaN,NaN,NaN,NaN,2P Meter,2P,NaN,NaN,NaN,NaN
4,1544,27052,1P MTR M-SAT 7:30-18:30,2P MTR M-SAT 18.30 - 20.30,1P SUN 7:30-18:30,NaN,NaN,NaN,120,240.0,...,0.0,NaN,NaN,NaN,1P Meter,2P,1P,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4258,2360,25816,1P MTR M-F 9:30-18:30,2P MTR M-SAT 18.30 - 20.30,1P MTR Sat 7.30am to 6.30pm,NaN,NaN,NaN,120,240.0,...,6.0,NaN,NaN,NaN,1P Meter,2P,1P Meter,NaN,NaN,NaN
4259,4326,23472,2P TKT A M-SAT 7:30-20:30,2P SUN 7:30-18:30,NaN,NaN,NaN,NaN,240,240.0,...,NaN,NaN,NaN,NaN,2P Ticket A,2P,NaN,NaN,NaN,NaN
4260,4323,23471,2P TKT A M-SAT 7:30-20:30,2P SUN 7:30-18:30,NaN,NaN,NaN,NaN,240,240.0,...,NaN,NaN,NaN,NaN,2P Ticket A,2P,NaN,NaN,NaN,NaN
4261,5331,29193,2P MTR M-F 7:30-18:30,2P MTR SAT 7:30-12:30,NaN,NaN,NaN,NaN,240,240.0,...,NaN,NaN,NaN,NaN,2P Meter,2P Meter,NaN,NaN,NaN,NaN


In [121]:
# ~24k rows have no KerbsideID — those bays haven't been assigned a sensor ID
show_missing(bays_raw, 'on-street-parking-bays')

  on-street-parking-bays  —  23981 rows with missing values
  Missing counts per column:
    KerbsideID                                     23981  (82.5%)



,RoadSegmentID,KerbsideID,RoadSegmentDescription,Latitude,Longitude,LastUpdated,Location
0,23259,NaN,South Wharf Drive between Foundry Way and Rigg...,-37.822453,144.937473,2025-09-03,"-37.8224529, 144.9374727"
1,21847,NaN,Spring Street between Little Lonsdale Street a...,-37.808065,144.971409,2025-09-03,"-37.8080651, 144.9714089"
2,22063,NaN,City Road between Southgate Avenue and Southba...,-37.822147,144.964801,2026-03-31,"-37.8221466, 144.9648011"
3,21618,NaN,Swanston Street between La Trobe Street and Li...,-37.809238,144.963572,2025-06-03,"-37.8092384, 144.9635721"
4,20935,NaN,Princess Street between Peel Street and Cobden...,-37.804961,144.956888,2025-06-03,"-37.8049609, 144.9568878"
...,...,...,...,...,...,...,...
29048,22618,NaN,Intersection of Albert Street and Morrison Place,-37.809693,144.976518,2026-01-15,"-37.809693, 144.9765185"
29049,20891,NaN,Albert Street between Morrison Place and Lansd...,-37.809827,144.977754,2026-01-15,"-37.8098274, 144.9777537"
29050,20891,NaN,Albert Street between Morrison Place and Lansd...,-37.809820,144.977686,2026-01-15,"-37.8098199, 144.9776862"
29051,20891,NaN,Albert Street between Morrison Place and Lansd...,-37.809722,144.976789,2026-01-15,"-37.8097223, 144.9767886"


In [122]:
# Zone_Number is null for some sensors (bay not linked to a zone)
show_missing(sensors_raw, 'on-street-parking-bay-sensors')

  on-street-parking-bay-sensors  —  240 rows with missing values
  Missing counts per column:
    Zone_Number                                      240  (7.1%)



,Lastupdated,Status_Timestamp,Zone_Number,Status_Description,KerbsideID,Location
19,2024-12-30T08:44:37+08:00,2024-03-31T20:33:16+08:00,NaN,Present,9351,"-37.80570432547628, 144.95951121701592"
21,2025-02-13T13:44:36+08:00,2023-11-29T07:35:32+08:00,NaN,Present,6934,"-37.82008789716498, 144.95730555908776"
29,2025-01-22T11:44:37+08:00,2024-05-03T11:56:57+08:00,NaN,Unoccupied,22776,"-37.82337001046017, 144.96661886495528"
33,2025-01-22T11:44:37+08:00,2024-05-03T12:21:09+08:00,NaN,Unoccupied,22772,"-37.823420177807634, 144.9668770118377"
34,2025-03-07T07:44:37+08:00,2025-03-07T06:35:37+08:00,NaN,Present,62713,"-37.81319239221083, 144.9704750954116"
...,...,...,...,...,...,...
3351,2026-06-28T16:31:39+08:00,2026-06-27T11:53:47+08:00,NaN,Present,61311,"-37.81800660511624, 144.96162588794874"
3364,2026-06-28T16:31:39+08:00,2026-06-28T15:14:27+08:00,NaN,Present,13463,"-37.80918921874, 144.9810107150118"
3366,2026-06-28T16:31:39+08:00,2026-06-28T16:08:10+08:00,NaN,Unoccupied,62212,"-37.817050192305175, 144.96090485850814"
3369,2026-06-28T16:31:39+08:00,2026-06-28T11:30:59+08:00,NaN,Unoccupied,63119,"-37.816172188777784, 144.96027917964432"


In [123]:
show_missing(zones_raw, 'parking-zones-linked-to-street-segments')

  parking-zones-linked-to-street-segments  —  5 rows with missing values
  Missing counts per column:
    StreetTo                                           5  (0.5%)



,ParkingZone,OnStreet,StreetFrom,StreetTo,Segment_ID
180,7226,Saint Mangos Lane North,Docklands Drive,NaN,23267
266,7978,Blakeney Place,Clarendon Street,NaN,22011
529,7182,Adela Lane,Collins Street,NaN,23342
862,7194,Enterprize Way,Bourke Street,NaN,22898
922,7997,Blakeney Place,Clarendon Street,NaN,22011


In [124]:
show_missing(paystay_raw, 'pay-stay-zones-linked-to-street-segments')

  pay-stay-zones-linked-to-street-segments  —  2 rows with missing values
  Missing counts per column:
    between_street_2                                   2  (0.2%)



,pay_stay_zone,street,between_street_1,between_street_2,street_segment_id
272,30001222,SEAFARER LANE,BOURKE STREET,NaN,23063
313,30001226,SAINT MANGOS LANE NORTH,DOCKLANDS DRIVE,NaN,23267


In [125]:
show_missing(signs_raw, 'sign-plates-located-in-each-parking-zone')

  sign-plates-located-in-each-parking-zone  —  0 rows with missing values
  No missing values.



### Duplicate rows — all files

In [168]:
show_duplicates(bayres_raw, 'on-street-car-park-bay-restrictions.csv')

# Same BatID
show_duplicates(
    bayres_raw,
    'on-street-car-park-bay-restrictions.csv',
    subset=['BayID']
)

  on-street-car-park-bay-restrictions.csv  —  0 duplicate rows (by all columns)
  No duplicates.

  on-street-car-park-bay-restrictions.csv  —  0 duplicate rows (by columns ['BayID'])
  No duplicates.



In [126]:
# Full-row duplicates
show_duplicates(bays_raw, 'on-street-parking-bays')

# Same KerbsideID on multiple rows (sensor ID collision)
show_duplicates(
    bays_raw.dropna(subset=['KerbsideID']),
    'on-street-parking-bays — duplicate KerbsideID',
    subset=['KerbsideID']
)

  on-street-parking-bays  —  0 duplicate rows (by all columns)
  No duplicates.

  on-street-parking-bays — duplicate KerbsideID  —  38 duplicate rows (by columns ['KerbsideID'])


,RoadSegmentID,KerbsideID,RoadSegmentDescription,Latitude,Longitude,LastUpdated,Location
2775,22630,17212,Wellington Parade South between Wellington Cre...,-37.816007,144.975320,2025-06-03,"-37.8160071, 144.9753201"
15845,20472,17212,Berkeley Street between Queensberry Street and...,-37.803806,144.959419,2025-09-03,"-37.8038058, 144.9594192"
12566,22630,17242,Wellington Parade South between Wellington Cre...,-37.816090,144.975931,2025-06-03,"-37.8160904, 144.975931"
6085,20472,17242,Berkeley Street between Queensberry Street and...,-37.802234,144.958695,2025-09-03,"-37.8022345, 144.9586945"
2767,22630,17243,Wellington Parade South between Wellington Cre...,-37.816086,144.975899,2025-06-03,"-37.816086, 144.975899"
22058,20472,17243,Berkeley Street between Queensberry Street and...,-37.802177,144.958668,2025-09-03,"-37.8021765, 144.9586678"
25841,22630,17244,Wellington Parade South between Wellington Cre...,-37.816081,144.975863,2025-06-03,"-37.8160807, 144.9758631"
6083,20472,17244,Berkeley Street between Queensberry Street and...,-37.803894,144.959309,2025-09-03,"-37.8038942, 144.9593095"
28141,22308,22415,Commercial Road between St Kilda Road and Punt...,-37.844948,144.980247,2025-06-03,"-37.8449479, 144.980247"
21745,20739,22415,Orr Street between Victoria Street and Earl St...,-37.806377,144.965173,2025-09-03,"-37.8063773, 144.9651734"


In [127]:
show_duplicates(sensors_raw, 'on-street-parking-bay-sensors')

# Same sensor appearing more than once (KerbsideID collision)
show_duplicates(sensors_raw, 'sensors — duplicate KerbsideID', subset=['KerbsideID'])

  on-street-parking-bay-sensors  —  0 duplicate rows (by all columns)
  No duplicates.

  sensors — duplicate KerbsideID  —  0 duplicate rows (by columns ['KerbsideID'])
  No duplicates.



In [128]:
show_duplicates(zones_raw, 'parking-zones')

# Same zone + segment pair appearing twice
show_duplicates(zones_raw, 'parking-zones — duplicate (Zone, Segment)',
                subset=['ParkingZone', 'Segment_ID'])

  parking-zones  —  0 duplicate rows (by all columns)
  No duplicates.

  parking-zones — duplicate (Zone, Segment)  —  0 duplicate rows (by columns ['ParkingZone', 'Segment_ID'])
  No duplicates.



In [129]:
show_duplicates(paystay_raw, 'pay-stay-zones')

  pay-stay-zones  —  0 duplicate rows (by all columns)
  No duplicates.



In [130]:
show_duplicates(signs_raw, 'sign-plates')

# Same zone + day + time window defined twice
show_duplicates(signs_raw, 'sign-plates — duplicate restriction rule',
                subset=['ParkingZone', 'Restriction_Days',
                        'Time_Restrictions_Start', 'Time_Restrictions_Finish'])

  sign-plates  —  0 duplicate rows (by all columns)
  No duplicates.

  sign-plates — duplicate restriction rule  —  44 duplicate rows (by columns ['ParkingZone', 'Restriction_Days', 'Time_Restrictions_Start', 'Time_Restrictions_Finish'])


,ParkingZone,Restriction_Days,Time_Restrictions_Start,Time_Restrictions_Finish,Restriction_Display
556,7007,Mon-Fri,16:00:00,19:00:00,MP3P
1746,7007,Mon-Fri,16:00:00,19:00:00,MP2P
558,7007,Mon-Fri,19:00:00,22:00:00,MP2P
1748,7007,Mon-Fri,19:00:00,22:00:00,MP3P
1143,7007,Sat-Sun,07:00:00,22:00:00,MP2P
1145,7007,Sat-Sun,07:00:00,22:00:00,MP3P
567,7012,Mon-Fri,07:00:00,19:00:00,MP2P
1152,7012,Mon-Fri,07:00:00,19:00:00,MP3P
578,7012,Mon-Fri,19:00:00,22:00:00,MP2P
734,7012,Mon-Fri,19:00:00,22:00:00,MP3P


**Ad-hoc filtering — useful one-liners:**
```python
# All sensors with no zone assigned
sensors_raw[sensors_raw['Zone_Number'].isnull()]

# Bays rows sharing the same KerbsideID
mask = bays_raw.duplicated(subset=['KerbsideID'], keep=False)
bays_raw[mask].sort_values('KerbsideID')

# Rows missing a specific column (e.g. Latitude)
bays_raw[bays_raw['Latitude'].isnull()]
```

In [153]:
bays[bays['KerbsideID'].astype(str).str.contains('[A-Za-z]', regex=True)]
# print(bays)

,RoadSegmentID,KerbsideID,RoadSegmentDescription,Latitude,Longitude,LastUpdated
1464,20951,7568N,Victoria Street between Capel Street and Howar...,-37.805646,144.954832,2025-06-03
11248,20951,7570N,Victoria Street between Capel Street and Howar...,-37.805639,144.954765,2025-06-03


---
## 1. On-Street Parking Bays (`on-street-parking-bays.csv`)
Physical bay locations — the master inventory.

In [132]:
print('Shape:', bays_raw.shape)
print('\nColumn dtypes:')
print(bays_raw.dtypes)
bays_raw.head()

Shape: (29053, 7)

Column dtypes:
RoadSegmentID               int64
KerbsideID                    str
RoadSegmentDescription        str
Latitude                  float64
Longitude                 float64
LastUpdated                   str
Location                      str
dtype: object


,RoadSegmentID,KerbsideID,RoadSegmentDescription,Latitude,Longitude,LastUpdated,Location
0,23259,NaN,South Wharf Drive between Foundry Way and Rigg...,-37.822453,144.937473,2025-09-03,"-37.8224529, 144.9374727"
1,21847,NaN,Spring Street between Little Lonsdale Street a...,-37.808065,144.971409,2025-09-03,"-37.8080651, 144.9714089"
2,22063,NaN,City Road between Southgate Avenue and Southba...,-37.822147,144.964801,2026-03-31,"-37.8221466, 144.9648011"
3,21618,NaN,Swanston Street between La Trobe Street and Li...,-37.809238,144.963572,2025-06-03,"-37.8092384, 144.9635721"
4,20935,NaN,Princess Street between Peel Street and Cobden...,-37.804961,144.956888,2025-06-03,"-37.8049609, 144.9568878"


In [133]:
print('=== Coordinate ranges ===')
print(bays_raw[['Latitude', 'Longitude']].describe().round(5))

=== Coordinate ranges ===
          Latitude    Longitude
count  29053.00000  29053.00000
mean     -37.80776    144.95628
std        0.01407      0.01806
min      -37.85054    144.90058
25%      -37.81632    144.94497
50%      -37.80594    144.95816
75%      -37.79672    144.96969
max      -37.77563    144.99063


In [134]:
bays = bays_raw.copy()
bays['LastUpdated'] = pd.to_datetime(bays['LastUpdated'], errors='coerce')
bays = bays.drop(columns=['Location'])

# Keep it as string so joins with the sensors table work correctly.
bays['KerbsideID'] = bays['KerbsideID'].astype(str).where(bays['KerbsideID'].notna(), other=pd.NA)

# Flag the non-numeric IDs so you can investigate them
non_numeric_kerb = bays[bays['KerbsideID'].notna() &
                        bays['KerbsideID'].str.contains('[A-Za-z]', regex=True)]
print(f'KerbsideIDs with letters (non-numeric): {len(non_numeric_kerb)}')
if len(non_numeric_kerb):
    display(non_numeric_kerb)

coord_null_mask = bays['Latitude'].isnull() | bays['Longitude'].isnull()
print(f'Rows dropped — missing coordinates: {coord_null_mask.sum()}')
bays = bays[~coord_null_mask]

out_of_bounds = bays[
    (bays['Latitude'] < -38.2) | (bays['Latitude'] > -37.5) |
    (bays['Longitude'] < 144.5) | (bays['Longitude'] > 145.5)
]
print(f'Rows outside Melbourne bounding box: {len(out_of_bounds)}')
if len(out_of_bounds):
    display(out_of_bounds)

before = len(bays)
bays = bays.drop_duplicates()
print(f'Duplicate rows removed: {before - len(bays)}')
print(f'\nCleaned bays shape: {bays.shape}')
bays.head()

KerbsideIDs with letters (non-numeric): 2


,RoadSegmentID,KerbsideID,RoadSegmentDescription,Latitude,Longitude,LastUpdated
1464,20951,7568N,Victoria Street between Capel Street and Howar...,-37.805646,144.954832,2025-06-03
11248,20951,7570N,Victoria Street between Capel Street and Howar...,-37.805639,144.954765,2025-06-03


Rows dropped — missing coordinates: 0
Rows outside Melbourne bounding box: 0
Duplicate rows removed: 0

Cleaned bays shape: (29053, 6)


,RoadSegmentID,KerbsideID,RoadSegmentDescription,Latitude,Longitude,LastUpdated
0,23259,NaN,South Wharf Drive between Foundry Way and Rigg...,-37.822453,144.937473,2025-09-03
1,21847,NaN,Spring Street between Little Lonsdale Street a...,-37.808065,144.971409,2025-09-03
2,22063,NaN,City Road between Southgate Avenue and Southba...,-37.822147,144.964801,2026-03-31
3,21618,NaN,Swanston Street between La Trobe Street and Li...,-37.809238,144.963572,2025-06-03
4,20935,NaN,Princess Street between Peel Street and Cobden...,-37.804961,144.956888,2025-06-03


In [173]:
bays1 = bays_raw.copy()
print(len(bays1))
bays_ids = bays1[bays1["KerbsideID"].notna()]
print(len(bays_ids))

29053
5072


---
## 2. Parking Bay Sensors (`on-street-parking-bay-sensors.csv`)
`Present` = car detected, `Unoccupied` = empty.

In [135]:
print('Shape:', sensors_raw.shape)
print('\nColumn dtypes:')
print(sensors_raw.dtypes)
sensors_raw.head()

Shape: (3388, 6)

Column dtypes:
Lastupdated               str
Status_Timestamp          str
Zone_Number           float64
Status_Description        str
KerbsideID              int64
Location                  str
dtype: object


,Lastupdated,Status_Timestamp,Zone_Number,Status_Description,KerbsideID,Location
0,2024-12-30T08:44:37+08:00,2024-08-18T16:23:46+08:00,7394.0,Unoccupied,9344,"-37.80494402936792, 144.95916129121264"
1,2025-01-10T10:44:36+08:00,2024-08-14T17:22:05+08:00,7392.0,Unoccupied,9373,"-37.80329074425858, 144.95836336946644"
2,2024-12-05T07:44:37+08:00,2024-11-28T10:30:57+08:00,7084.0,Present,8735,"-37.80223335664963, 144.96120480793184"
3,2024-12-05T07:44:37+08:00,2024-11-28T07:23:40+08:00,7084.0,Unoccupied,8749,"-37.80230361629087, 144.9618505999636"
4,2025-01-09T13:44:36+08:00,2023-08-21T08:09:03+08:00,7800.0,Present,24505,"-37.79756250415984, 144.95759881813447"


In [136]:
print('=== Status values ===')
print(sensors_raw['Status_Description'].value_counts())

print('\n=== Timestamp ranges ===')
for col in ['Lastupdated', 'Status_Timestamp']:
    ts = pd.to_datetime(sensors_raw[col], errors='coerce')
    print(f'  {col}: {ts.min()} → {ts.max()}')

=== Status values ===
Status_Description
Present       1978
Unoccupied    1410
Name: count, dtype: int64

=== Timestamp ranges ===
  Lastupdated: 2024-12-05 07:44:37+08:00 → 2026-06-28 16:31:39+08:00
  Status_Timestamp: 2022-09-13 12:38:23+08:00 → 2026-06-28 16:28:54+08:00


In [137]:
sensors = sensors_raw.copy()
for col in ['Lastupdated', 'Status_Timestamp']:
    sensors[col] = pd.to_datetime(sensors[col], utc=True, errors='coerce')

sensors['Zone_Number'] = pd.to_numeric(sensors['Zone_Number'], errors='coerce').astype('Int64')
sensors = sensors.drop(columns=['Location'])
sensors['is_occupied'] = sensors['Status_Description'].str.strip() == 'Present'

null_ts = sensors['Status_Timestamp'].isnull().sum()
print(f'Rows with null Status_Timestamp: {null_ts}')
sensors = sensors[sensors['Status_Timestamp'].notna()]

before = len(sensors)
sensors = sensors.drop_duplicates()
print(f'Duplicate rows removed: {before - len(sensors)}')
print(f'\nCleaned sensors shape: {sensors.shape}')
print(f'Overall occupancy rate: {sensors["is_occupied"].mean():.1%}')
sensors.head()

Rows with null Status_Timestamp: 0
Duplicate rows removed: 0

Cleaned sensors shape: (3388, 6)
Overall occupancy rate: 58.4%


,Lastupdated,Status_Timestamp,Zone_Number,Status_Description,KerbsideID,is_occupied
0,2024-12-30 00:44:37+00:00,2024-08-18 08:23:46+00:00,7394,Unoccupied,9344,False
1,2025-01-10 02:44:36+00:00,2024-08-14 09:22:05+00:00,7392,Unoccupied,9373,False
2,2024-12-04 23:44:37+00:00,2024-11-28 02:30:57+00:00,7084,Present,8735,True
3,2024-12-04 23:44:37+00:00,2024-11-27 23:23:40+00:00,7084,Unoccupied,8749,False
4,2025-01-09 05:44:36+00:00,2023-08-21 00:09:03+00:00,7800,Present,24505,True


---
## 3. Parking Zones → Street Segments

In [138]:
print('Shape:', zones_raw.shape)
print(zones_raw.dtypes)
zones_raw.head()

Shape: (926, 5)
ParkingZone    int64
OnStreet         str
StreetFrom       str
StreetTo         str
Segment_ID     int64
dtype: object


,ParkingZone,OnStreet,StreetFrom,StreetTo,Segment_ID
0,7369,Dallas Brooks Drive,Domain Road,Birdwood Avenue,22469
1,7075,Lygon Street,Argyle Place North,Grattan Street,20530
2,7019,Bouverie Street,Victoria Street,Queensberry Street,20462
3,7076,Lygon Street,Victoria Street,Queensberry Street,20522
4,7720,Harcourt Street,Courtney Street,Flemington Road,21180


In [139]:
print('=== Zones spanning multiple segments ===')
multi = zones_raw.groupby('ParkingZone')['Segment_ID'].nunique()
print(multi[multi > 1].sort_values(ascending=False).head(10))

=== Zones spanning multiple segments ===
ParkingZone
7950    5
7797    4
7776    4
7976    3
7804    3
7258    3
7876    3
7785    3
7781    3
7348    3
Name: Segment_ID, dtype: int64


In [140]:
zones = zones_raw.copy()
before = len(zones)
zones = zones.drop_duplicates()
print(f'Duplicate rows removed: {before - len(zones)}')

str_cols = zones.select_dtypes(include=['object', 'string']).columns
zones[str_cols] = zones[str_cols].apply(lambda c: c.str.strip())
for col in ['OnStreet', 'StreetFrom', 'StreetTo']:
    zones[col] = zones[col].str.title()

print(f'Cleaned zones shape: {zones.shape}')
zones.head()

Duplicate rows removed: 0
Cleaned zones shape: (926, 5)


,ParkingZone,OnStreet,StreetFrom,StreetTo,Segment_ID
0,7369,Dallas Brooks Drive,Domain Road,Birdwood Avenue,22469
1,7075,Lygon Street,Argyle Place North,Grattan Street,20530
2,7019,Bouverie Street,Victoria Street,Queensberry Street,20462
3,7076,Lygon Street,Victoria Street,Queensberry Street,20522
4,7720,Harcourt Street,Courtney Street,Flemington Road,21180


---
## 4. Pay-and-Stay Zones → Street Segments

In [141]:
print('Shape:', paystay_raw.shape)
print(paystay_raw.dtypes)
paystay_raw.head()

Shape: (935, 5)
pay_stay_zone        int64
street                 str
between_street_1       str
between_street_2       str
street_segment_id    int64
dtype: object


,pay_stay_zone,street,between_street_1,between_street_2,street_segment_id
0,30001023,LEICESTER STREET,VICTORIA STREET,BERKELEY STREET,20453
1,30001026,BOUVERIE STREET,VICTORIA STREET,QUEENSBERRY STREET,20462
2,30001027,BOUVERIE STREET,QUEENSBERRY STREET,VICTORIA STREET,20462
3,30001028,CARDIGAN STREET,ARGYLE PLACE NORTH,GRATTAN STREET,20512
4,30001029,CARDIGAN STREET,GRATTAN STREET,ARGYLE PLACE NORTH,20512


In [142]:
paystay = paystay_raw.copy()
paystay = paystay.rename(columns={
    'pay_stay_zone': 'PayStayZone',
    'street': 'OnStreet',
    'between_street_1': 'StreetFrom',
    'between_street_2': 'StreetTo',
    'street_segment_id': 'Segment_ID'
})
before = len(paystay)
paystay = paystay.drop_duplicates()
print(f'Duplicate rows removed: {before - len(paystay)}')
for col in ['OnStreet', 'StreetFrom', 'StreetTo']:
    paystay[col] = paystay[col].str.title()
print(f'Cleaned pay-stay shape: {paystay.shape}')
paystay.head()

Duplicate rows removed: 0
Cleaned pay-stay shape: (935, 5)


,PayStayZone,OnStreet,StreetFrom,StreetTo,Segment_ID
0,30001023,Leicester Street,Victoria Street,Berkeley Street,20453
1,30001026,Bouverie Street,Victoria Street,Queensberry Street,20462
2,30001027,Bouverie Street,Queensberry Street,Victoria Street,20462
3,30001028,Cardigan Street,Argyle Place North,Grattan Street,20512
4,30001029,Cardigan Street,Grattan Street,Argyle Place North,20512


---
## 5. Sign Plates / Restrictions

In [143]:
print('Shape:', signs_raw.shape)
signs_raw.head(10)

Shape: (1821, 5)


,ParkingZone,Restriction_Days,Time_Restrictions_Start,Time_Restrictions_Finish,Restriction_Display
0,7500,Mon-Fri,07:00:00,19:00:00,MP2P
1,7502,Mon-Fri,16:00:00,19:00:00,MP2P
2,7177,Mon-Fri,07:00:00,19:00:00,MP2P
3,7176,Sat-Sun,07:00:00,22:00:00,MP2P
4,7176,Mon-Fri,16:00:00,19:00:00,MP2P
5,7176,Mon-Fri,07:00:00,16:00:00,LZ30
6,7690,Sat-Sun,07:00:00,22:00:00,MP2P
7,7690,Mon-Fri,07:00:00,16:00:00,LZ30
8,7689,Sat-Sun,07:00:00,22:00:00,MP2P
9,7025,Mon-Fri,07:00:00,19:00:00,MP2P


In [144]:
print('=== Restriction types ===')
print(signs_raw['Restriction_Display'].value_counts().head(20))
print('\n=== Day patterns ===')
print(signs_raw['Restriction_Days'].value_counts())

=== Restriction types ===
Restriction_Display
MP2P    1262
LZ30     194
2P       118
MP3P      51
4P        48
PP        42
MP4P      39
3P        17
1P        15
QP         8
FP15       8
MP1P       8
SP         4
DP2P       3
5P         2
FP2P       1
FP1P       1
Name: count, dtype: int64

=== Day patterns ===
Restriction_Days
Mon-Fri    1214
Sat-Sun     456
Sat          72
Mon-Sat      67
Mon-Sun       8
Sun           2
Mon-Thu       1
Fri           1
Name: count, dtype: int64


In [145]:
signs = signs_raw.copy()
for col in ['Time_Restrictions_Start', 'Time_Restrictions_Finish']:
    signs[col] = pd.to_timedelta(signs[col], errors='coerce')

time_issues = signs[signs['Time_Restrictions_Finish'] <= signs['Time_Restrictions_Start']]
print(f'Rows where finish <= start: {len(time_issues)}')
if len(time_issues):
    display(time_issues)

str_cols = signs.select_dtypes(include=['object', 'string']).columns
signs[str_cols] = signs[str_cols].apply(lambda c: c.str.strip())
before = len(signs)
signs = signs.drop_duplicates()
print(f'Duplicate rows removed: {before - len(signs)}')
print(f'Cleaned signs shape: {signs.shape}')
signs.head()

Rows where finish <= start: 0
Duplicate rows removed: 0
Cleaned signs shape: (1821, 5)


,ParkingZone,Restriction_Days,Time_Restrictions_Start,Time_Restrictions_Finish,Restriction_Display
0,7500,Mon-Fri,0 days 07:00:00,0 days 19:00:00,MP2P
1,7502,Mon-Fri,0 days 16:00:00,0 days 19:00:00,MP2P
2,7177,Mon-Fri,0 days 07:00:00,0 days 19:00:00,MP2P
3,7176,Sat-Sun,0 days 07:00:00,0 days 22:00:00,MP2P
4,7176,Mon-Fri,0 days 16:00:00,0 days 19:00:00,MP2P


---
## 6. Cross-Table Checks

In [146]:
zone_ids      = set(zones['ParkingZone'].unique())
sign_zone_ids = set(signs['ParkingZone'].unique())
sensor_zones  = set(sensors['Zone_Number'].dropna().astype(int).unique())

print('Unique zone IDs in zones table:       ', len(zone_ids))
print('Unique zone IDs in signs table:       ', len(sign_zone_ids))
print('Unique zone IDs reported by sensors:  ', len(sensor_zones))
print('\nZones with signs but no street mapping:', len(sign_zone_ids - zone_ids))
print('Zones with street mapping but no signs:', len(zone_ids - sign_zone_ids))
print('Sensor zones not in zones table:       ', len(sensor_zones - zone_ids))

bay_kerbside    = set(bays['KerbsideID'].dropna().astype(str).unique())
sensor_kerbside = set(sensors['KerbsideID'].astype(str).unique())
print('\nUnique KerbsideIDs in bays table:    ', len(bay_kerbside))
print('Unique KerbsideIDs in sensors table: ', len(sensor_kerbside))
print('Sensor KerbsideIDs not in bays:      ', len(sensor_kerbside - bay_kerbside))

Unique zone IDs in zones table:        839
Unique zone IDs in signs table:        622
Unique zone IDs reported by sensors:   341

Zones with signs but no street mapping: 1
Zones with street mapping but no signs: 218
Sensor zones not in zones table:        7

Unique KerbsideIDs in bays table:     5053
Unique KerbsideIDs in sensors table:  3388
Sensor KerbsideIDs not in bays:       1039


---
## 7. Build Unified Analysis Table

Join chain (based on actual key overlap):

```
signs ──(ParkingZone)──► zones ──(Segment_ID = RoadSegmentID)──► bays
                           │                                       │
               (ParkingZone = Zone_Number)              (KerbsideID)
                           │                                       │
                        sensors ◄──────────────────────────────────┘

paystay ──(Segment_ID)──► zones   [parallel lookup, not in main chain]
```

**Step 1** — zones + signs: attach restriction rules to each zone  
**Step 2** — zones_with_signs + bays: attach bay coordinates via `Segment_ID = RoadSegmentID`  
**Step 3** — sensors + bays: attach precise coordinates via `KerbsideID` (where available)  
**Step 4** — sensors + zones_with_signs: attach zone/street/restriction context via `Zone_Number`  
**Step 5** — zones + paystay: check which zones also have pay-and-stay coverage

In [147]:
# ── Step 1: zones + signs ──────────────────────────────────────────────────
# A zone can have multiple sign rows (different day/time windows), so this is
# a one-to-many join. Keep all restriction rows per zone.
zones_with_signs = zones.merge(
    signs[['ParkingZone', 'Restriction_Days',
           'Time_Restrictions_Start', 'Time_Restrictions_Finish',
           'Restriction_Display']],
    on='ParkingZone',
    how='left'   # keep zones even if they have no sign data
)

print('zones_with_signs shape:', zones_with_signs.shape)
print(f"  zones with at least one restriction: "
      f"{zones_with_signs[zones_with_signs['Restriction_Display'].notna()]['ParkingZone'].nunique()}")
print(f"  zones with no restriction data:      "
      f"{zones_with_signs[zones_with_signs['Restriction_Display'].isna()]['ParkingZone'].nunique()}")
zones_with_signs.head()

zones_with_signs shape: (2203, 9)
  zones with at least one restriction: 621
  zones with no restriction data:      218


,ParkingZone,OnStreet,StreetFrom,StreetTo,Segment_ID,Restriction_Days,Time_Restrictions_Start,Time_Restrictions_Finish,Restriction_Display
0,7369,Dallas Brooks Drive,Domain Road,Birdwood Avenue,22469,Sat-Sun,0 days 07:00:00,0 days 22:00:00,MP4P
1,7369,Dallas Brooks Drive,Domain Road,Birdwood Avenue,22469,Mon-Fri,0 days 19:00:00,0 days 22:00:00,MP4P
2,7369,Dallas Brooks Drive,Domain Road,Birdwood Avenue,22469,Mon-Fri,0 days 07:00:00,0 days 19:00:00,MP4P
3,7075,Lygon Street,Argyle Place North,Grattan Street,20530,NaN,NaT,NaT,NaN
4,7019,Bouverie Street,Victoria Street,Queensberry Street,20462,Mon-Fri,0 days 07:00:00,0 days 16:00:00,LZ30


In [148]:
# ── Step 2: zones_with_signs + bays via Segment_ID = RoadSegmentID ─────────
# Each zone segment maps to many individual bays (bays are points on the road).
# This expands the table: one zone row → many bay coordinate rows.
bays_for_join = bays[['RoadSegmentID', 'KerbsideID',
                       'Latitude', 'Longitude']].copy()
bays_for_join = bays_for_join.rename(columns={'RoadSegmentID': 'Segment_ID'})

zones_bays = zones_with_signs.merge(
    bays_for_join,
    on='Segment_ID',
    how='left'
)

print('zones_bays shape:', zones_bays.shape)
print('Zones matched to at least one bay:',
      zones_bays[zones_bays['Latitude'].notna()]['ParkingZone'].nunique())
print('Zones with no bay match:          ',
      zones_bays[zones_bays['Latitude'].isna()]['ParkingZone'].nunique())
zones_bays.head()

zones_bays shape: (69163, 12)
Zones matched to at least one bay: 838
Zones with no bay match:           2


,ParkingZone,OnStreet,StreetFrom,StreetTo,Segment_ID,Restriction_Days,Time_Restrictions_Start,Time_Restrictions_Finish,Restriction_Display,KerbsideID,Latitude,Longitude
0,7369,Dallas Brooks Drive,Domain Road,Birdwood Avenue,22469,Sat-Sun,0 days 07:00:00,0 days 22:00:00,MP4P,NaN,-37.831674,144.976623
1,7369,Dallas Brooks Drive,Domain Road,Birdwood Avenue,22469,Sat-Sun,0 days 07:00:00,0 days 22:00:00,MP4P,NaN,-37.831728,144.976634
2,7369,Dallas Brooks Drive,Domain Road,Birdwood Avenue,22469,Sat-Sun,0 days 07:00:00,0 days 22:00:00,MP4P,NaN,-37.831892,144.976673
3,7369,Dallas Brooks Drive,Domain Road,Birdwood Avenue,22469,Sat-Sun,0 days 07:00:00,0 days 22:00:00,MP4P,NaN,-37.832060,144.976692
4,7369,Dallas Brooks Drive,Domain Road,Birdwood Avenue,22469,Sat-Sun,0 days 07:00:00,0 days 22:00:00,MP4P,NaN,-37.832228,144.976659


In [149]:
# ── Step 3: sensors enriched with precise bay coordinates via KerbsideID ───
# Only ~69% of sensors match a bay KerbsideID — the rest will get coordinates
# from the zone-level join in step 4.
bays_coords = bays[bays['KerbsideID'].notna()][[
    'KerbsideID', 'RoadSegmentID', 'Latitude', 'Longitude'
]].copy()
bays_coords['KerbsideID'] = bays_coords['KerbsideID'].astype(str)
sensors['KerbsideID'] = sensors['KerbsideID'].astype(str)

sensors_with_coords = sensors.merge(
    bays_coords.rename(columns={
        'Latitude': 'Bay_Latitude',
        'Longitude': 'Bay_Longitude'
    }),
    on='KerbsideID',
    how='left'
)

matched = sensors_with_coords['Bay_Latitude'].notna().sum()
print(f'Sensors with precise bay coordinates: {matched} / {len(sensors_with_coords)}')
sensors_with_coords.head()

Sensors with precise bay coordinates: 2368 / 3407


,Lastupdated,Status_Timestamp,Zone_Number,Status_Description,KerbsideID,is_occupied,RoadSegmentID,Bay_Latitude,Bay_Longitude
0,2024-12-30 00:44:37+00:00,2024-08-18 08:23:46+00:00,7394,Unoccupied,9344,False,NaN,NaN,NaN
1,2025-01-10 02:44:36+00:00,2024-08-14 09:22:05+00:00,7392,Unoccupied,9373,False,NaN,NaN,NaN
2,2024-12-04 23:44:37+00:00,2024-11-28 02:30:57+00:00,7084,Present,8735,True,NaN,NaN,NaN
3,2024-12-04 23:44:37+00:00,2024-11-27 23:23:40+00:00,7084,Unoccupied,8749,False,NaN,NaN,NaN
4,2025-01-09 05:44:36+00:00,2023-08-21 00:09:03+00:00,7800,Present,24505,True,NaN,NaN,NaN


In [150]:
# ── Step 4: attach zone + street + restriction context via Zone_Number ──────
# Build a zone-level lookup (one row per zone — take the first/primary segment
# and pick a representative coordinate by averaging bay lat/lons per zone).
zone_coords = zones_bays.groupby('ParkingZone').agg(
    OnStreet=('OnStreet', 'first'),
    StreetFrom=('StreetFrom', 'first'),
    StreetTo=('StreetTo', 'first'),
    Segment_ID=('Segment_ID', 'first'),
    Zone_Latitude=('Latitude', 'mean'),
    Zone_Longitude=('Longitude', 'mean')
).reset_index()

# Attach restriction summary per zone (concat all restriction types)
restriction_summary = (
    signs.groupby('ParkingZone')['Restriction_Display']
    .apply(lambda x: ', '.join(sorted(x.dropna().unique())))
    .reset_index()
    .rename(columns={'Restriction_Display': 'Restriction_Types'})
)
zone_coords = zone_coords.merge(restriction_summary, on='ParkingZone', how='left')

# Now join onto sensors
sensors_full = sensors_with_coords.merge(
    zone_coords.rename(columns={'ParkingZone': 'Zone_Number'}),
    on='Zone_Number',
    how='left'
)

# Use precise bay coordinate if available, fall back to zone centroid
sensors_full['Latitude'] = sensors_full['Bay_Latitude'].fillna(sensors_full['Zone_Latitude'])
sensors_full['Longitude'] = sensors_full['Bay_Longitude'].fillna(sensors_full['Zone_Longitude'])

print('sensors_full shape:', sensors_full.shape)
print('\nCoordinate coverage:')
print(f"  Precise (KerbsideID match): {sensors_full['Bay_Latitude'].notna().sum()}")
print(f"  Zone centroid fallback:     {(sensors_full['Bay_Latitude'].isna() & sensors_full['Zone_Latitude'].notna()).sum()}")
print(f"  No coordinates at all:      {sensors_full['Latitude'].isna().sum()}")
print('\nKey columns null count:')
print(sensors_full[['Latitude', 'Longitude', 'OnStreet', 'Zone_Number', 'Restriction_Types']].isnull().sum())
sensors_full.head()

sensors_full shape: (3407, 18)

Coordinate coverage:
  Precise (KerbsideID match): 2368
  Zone centroid fallback:     922
  No coordinates at all:      117

Key columns null count:
Latitude             117
Longitude            117
OnStreet             301
Zone_Number          244
Restriction_Types    557
dtype: int64


,Lastupdated,Status_Timestamp,Zone_Number,Status_Description,KerbsideID,is_occupied,RoadSegmentID,Bay_Latitude,Bay_Longitude,OnStreet,StreetFrom,StreetTo,Segment_ID,Zone_Latitude,Zone_Longitude,Restriction_Types,Latitude,Longitude
0,2024-12-30 00:44:37+00:00,2024-08-18 08:23:46+00:00,7394,Unoccupied,9344,False,NaN,NaN,NaN,Elizabeth Street,Victoria Street,Queensberry Street,21113.0,-37.805028,144.958953,MP2P,-37.805028,144.958953
1,2025-01-10 02:44:36+00:00,2024-08-14 09:22:05+00:00,7392,Unoccupied,9373,False,NaN,NaN,NaN,Elizabeth Street,Queensberry Street,Pelham Street,21115.0,-37.802948,144.957942,"LZ30, MP3P",-37.802948,144.957942
2,2024-12-04 23:44:37+00:00,2024-11-28 02:30:57+00:00,7084,Present,8735,True,NaN,NaN,NaN,Pelham Street,Bouverie Street,Leicester Street,20873.0,-37.802235,144.961383,MP2P,-37.802235,144.961383
3,2024-12-04 23:44:37+00:00,2024-11-27 23:23:40+00:00,7084,Unoccupied,8749,False,NaN,NaN,NaN,Pelham Street,Bouverie Street,Leicester Street,20873.0,-37.802235,144.961383,MP2P,-37.802235,144.961383
4,2025-01-09 05:44:36+00:00,2023-08-21 00:09:03+00:00,7800,Present,24505,True,NaN,NaN,NaN,Royal Parade,Grattan Street,Story Street,22514.0,-37.797871,144.957795,MP2P,-37.797871,144.957795


In [151]:
# ── Step 5: zones + paystay — which zones also have pay-and-stay? ───────────
paystay_segs = set(paystay['Segment_ID'].unique())
zones['has_paystay'] = zones['Segment_ID'].isin(paystay_segs)

print('Zones with pay-and-stay coverage:', zones['has_paystay'].sum())
print('Zones without:                   ', (~zones['has_paystay']).sum())
zones[zones['has_paystay']].head()

Zones with pay-and-stay coverage: 802
Zones without:                    124


,ParkingZone,OnStreet,StreetFrom,StreetTo,Segment_ID,has_paystay
0,7369,Dallas Brooks Drive,Domain Road,Birdwood Avenue,22469,True
1,7075,Lygon Street,Argyle Place North,Grattan Street,20530,True
2,7019,Bouverie Street,Victoria Street,Queensberry Street,20462,True
3,7076,Lygon Street,Victoria Street,Queensberry Street,20522,True
4,7720,Harcourt Street,Courtney Street,Flemington Road,21180,True


---
## 8. Export Cleaned Files

In [152]:
import os
out_dir = '../data/cleaned'
os.makedirs(out_dir, exist_ok=True)

# Individual cleaned files
bays.to_csv(f'{out_dir}/bays_clean.csv', index=False)
sensors.to_csv(f'{out_dir}/sensors_clean.csv', index=False)
zones.to_csv(f'{out_dir}/zones_clean.csv', index=False)
paystay.to_csv(f'{out_dir}/paystay_clean.csv', index=False)
signs.to_csv(f'{out_dir}/signs_clean.csv', index=False)

# Joined tables
zones_with_signs.to_csv(f'{out_dir}/zones_with_signs.csv', index=False)
zones_bays.to_csv(f'{out_dir}/zones_bays.csv', index=False)
sensors_full.to_csv(f'{out_dir}/sensors_full.csv', index=False)

print('Exported to', out_dir)
exports = [
    ('bays_clean',       bays),
    ('sensors_clean',    sensors),
    ('zones_clean',      zones),
    ('paystay_clean',    paystay),
    ('signs_clean',      signs),
    ('zones_with_signs', zones_with_signs),
    ('zones_bays',       zones_bays),
    ('sensors_full',     sensors_full),
]
for name, df in exports:
    print(f'  {name+".csv":<30} {len(df):>6} rows  x  {df.shape[1]} cols')

Exported to ../data/cleaned
  bays_clean.csv                  29053 rows  x  6 cols
  sensors_clean.csv                3388 rows  x  6 cols
  zones_clean.csv                   926 rows  x  6 cols
  paystay_clean.csv                 935 rows  x  5 cols
  signs_clean.csv                  1821 rows  x  5 cols
  zones_with_signs.csv             2203 rows  x  9 cols
  zones_bays.csv                  69163 rows  x  12 cols
  sensors_full.csv                 3407 rows  x  18 cols


In [162]:
from collections import Counter

c1 = Counter(zones_raw['Segment_ID'].astype(str))
c2 = Counter(bays_raw['RoadSegmentID'].astype(str))
c3 = Counter(paystay_raw['street_segment_id'].astype(str))

common = sum(min(c1[k], c2[k]) for k in c1.keys() & c3.keys())

print(common)

801


In [1]:
import sys
from pathlib import Path

# Add the backend folder to Python's search path
sys.path.append(str(Path("../backend").resolve()))

from data_pipeline import get_full_dataset

full = get_full_dataset()

print(full.columns.tolist())

INFO  Sensors: fetching from API…
INFO    sensors: fetched 100 / 3390 records
INFO    sensors: fetched 200 / 3390 records
INFO    sensors: fetched 300 / 3390 records
INFO    sensors: fetched 400 / 3390 records
INFO    sensors: fetched 500 / 3390 records
INFO    sensors: fetched 600 / 3390 records
INFO    sensors: fetched 700 / 3390 records
INFO    sensors: fetched 800 / 3390 records
INFO    sensors: fetched 900 / 3390 records
INFO    sensors: fetched 1000 / 3390 records
INFO    sensors: fetched 1100 / 3390 records
INFO    sensors: fetched 1200 / 3390 records
INFO    sensors: fetched 1300 / 3390 records
INFO    sensors: fetched 1400 / 3390 records
INFO    sensors: fetched 1500 / 3390 records
INFO    sensors: fetched 1600 / 3390 records
INFO    sensors: fetched 1700 / 3390 records
INFO    sensors: fetched 1800 / 3390 records
INFO    sensors: fetched 1900 / 3390 records
INFO    sensors: fetched 2000 / 3390 records
INFO    sensors: fetched 2100 / 3390 records
INFO    sensors: fetched 2200 

['roadsegmentid', 'kerbsideid', 'roadsegmentdescription', 'latitude', 'longitude', 'lastupdated', 'onstreet', 'streetfrom', 'streetto', 'restriction_types', 'restrictions_json', 'has_paystay', 'zone_number', 'is_occupied', 'status_timestamp', 'has_sensor']


In [6]:
import pandas as pd
signs = pd.read_parquet("../data/cache/signs.parquet")
print("All unique restriction_days values:")
print(signs["restriction_days"].value_counts().to_string())

All unique restriction_days values:
restriction_days
Mon-Fri    1214
Sat-Sun     456
Sat          72
Mon-Sat      67
Mon-Sun       8
Sun           2
Mon-Thu       1
Fri           1
